# BART Ridership — Kaggle Çalışma Notebook'u

Bu notebook **Kaggle'da çalıştırılır**. Asıl mantık modüler `src/` kodundadır ve
GitHub'dan `clone` ile gelir. Projedeki her adım (Adım 2, 3, 4 ...) **sırayla bu
notebook'a eklenir** — üst üste birikir.

> İş bölümü: `src/` ve yerel scriptler repo'da (yerelde doğrulanır). Bu notebook
> sadece **Kaggle tarafıdır** (clone + import + çalıştır).

## 0. Kurulum — repo clone + import yolu

Notebook ayarları (sağ panel): **Internet → On** (clone için zorunlu),
**Accelerator → GPU** (Adım 4 eğitiminden itibaren; Adım 2 için gerekmez).

In [ ]:
import shutil, os, sys

REPO = "bay-area-transit-ridership-forecasting"
URL  = "https://github.com/osman-ozcanli/bay-area-transit-ridership-forecasting.git"

# 1) Her zaman taze kod: eskiyi sil, yeniden clone et
shutil.rmtree(REPO, ignore_errors=True)
os.system(f"git clone -q {URL}")

# 2) Import yolunu ekle
repo_path = f"/kaggle/working/{REPO}"
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

# 3) Eski cache'lenmis src modullerini temizle -> kernel restart gerekmeden taze import
for _m in list(sys.modules):
    if _m == "src" or _m.startswith("src."):
        del sys.modules[_m]

# 4) Self-check: clone gercekten geldi mi? (Internet kapali / push yapilmadiysa uyarir)
train_py = os.path.join(REPO, "src", "models", "train.py")
assert os.path.exists(train_py), "CLONE BASARISIZ -> Internet acik mi? GitHub push yapildi mi?"
print("setup ok. src/models ->", os.listdir(os.path.join(REPO, "src", "models")))

## Adım 2 — Veri Yükleme (tam veri)

Kaggle'da `use_sample=False` ile **tam ~13.3M satır** yüklenir (yereldeki 1.12M'lik
örnek değil). `/kaggle/input/bart-ridership/` altında 3 CSV ekli olmalı.

`src/data/load.py` içindeki `load_dataset`: 2016+2017 birleştir → DateTime parse →
istasyon merge → WSPR ekle.

In [ ]:
from src.config import load_config
from src.data.load import load_dataset

cfg = load_config()
cfg["environment"] = "kaggle"   # /kaggle/input path'leri
cfg["use_sample"]  = False      # tam veri (~13.3M)

df = load_dataset(cfg)
print("shape:", df.shape)
print("date range:", df["DateTime"].min(), "->", df["DateTime"].max())
df.head(3)

## Adım 3 — Feature Engineering

`src/features/build_features.py`: zaman (Hour/DayOfWeek/Month/IsWeekend/Period) +
tatil (CA, IsHoliday) + mesafe (haversine dist_km) feature'ları, **self-trip drop**
(analiz 3.5), **lag/rolling** (leakage-safe, OD-bazlı) ve LightGBM için **category
dtype** (cat.codes ordinal fix, analiz 3.3).

> Not: Tam veride (~13.3M) groupby/rolling işlemi biraz sürebilir.

In [ ]:
from src.features.build_features import build_features

df = build_features(df, cfg)

print("shape:", df.shape)
lag_cols = [c for c in df.columns if "lag" in c or "roll" in c]
print("lag/rolling kolonlari:", lag_cols)
print("self-trip kalan:", int((df["Origin"] == df["Destination"]).sum()))
print("IsHoliday orani:", round(df["IsHoliday"].mean(), 4))
df.head(3)

## Adım 4 — Temporal Split + LightGBM (GPU) + Model Kayıt

Bu hücre modeli **gerçekten eğitir**: 2016 train / 2017 test (temporal split,
leakage yok), TimeSeriesSplit CV (varyans), LightGBM categorical_feature + GPU,
early stopping; sonra `models/bart_lgb_final.txt`'e kaydeder.

> Süre ~65-70 dk (CV dahil). Eğitmeden devam etmek istersen aşağıdaki **(Bilgi)**
> notundaki 'modeli yükle' yolunu kullan.

In [ ]:
from src.models.train import train_model, save_model

cfg["model"]["device"] = "gpu"   # Kaggle GPU
model, results = train_model(df, cfg)

m = results['metrics']
print("device       :", results["device"])
print("train / test :", f"{results['n_train']:,} / {results['n_test']:,}")
print("best_iter    :", results["best_iteration"])
print("holdout MAE  :", round(m["mae"], 4))
print("holdout RMSE :", round(m["rmse"], 4))
print("holdout R2   :", round(m["r2"], 4))
if "cv" in results:
    cv = results["cv"]
    print("CV MAE       :", round(cv["mean_mae"], 4), "+/-", round(cv["std_mae"], 4))

save_model(model, cfg)

### (Bilgi) Eğitim yerine modeli YÜKLEMEK istersen

Model repoda kayıtlı (`models/bart_lgb_final.txt`, clone getirir). Yukarıdaki 70 dk'lık
eğitimi tekrarlamadan devam etmek için, aşağıyı bir **kod** hücresine çevir:

```python
import os, lightgbm as lgb
from src.config import PROJECT_ROOT
model_path = os.path.join(str(PROJECT_ROOT), "models", cfg["files"]["final_model"])
model = lgb.Booster(model_file=model_path)
print("Model yuklendi:", model_path)
```

## Adım 5 — Değerlendirme + Feature Importance YORUMU

Yüklenen modeli 2017 holdout'ta değerlendiriyoruz, feature importance çıkarıp
**yorumluyoruz** (analiz 3.7: 'grafik vardı, yorum yoktu' eksikliği).

### Bulgular (tam veri)
- **`Throughput_lag_1` tek başına gain'in ~%63'ü** → bir önceki saatin yolcu sayısı,
  gelecek saati belirleyen en güçlü sinyal. Talep güçlü **otokorelasyonlu**.
- **Lag + rolling birlikte ~%69** → zaman-serisi feature'ları (senior eklememiz)
  modelin belkemiği.
- Kalan: **Hour (~%9), Origin (~%7), Destination (~%6), Period (~%5)** → 'ne zaman +
  nerede' yapısı; EDA'daki yoğun saat (rush hour) ve yoğun istasyon (EMBR/MONT)
  bulgularıyla tutarlı.
- `dist_km` ~%2, gün/ay/tatil minik → mesafe sabit OD yapısında zaten Origin/Destination'a gömülü.
- **Senior not:** lag_1 baskınlığı modelin 'persistence + düzeltme' karakterini gösterir;
  lag olmayan cold-start tahmininde Hour/Origin/Destination öne çıkardı.

In [ ]:
from src.models.evaluate import (evaluate_holdout, feature_importance_df,
                                 plot_feature_importance, plot_residuals, interpret_importance)
from src.config import get_paths

metrics, y_true, y_pred = evaluate_holdout(model, df, cfg)
print("Holdout 2017 -> MAE:", round(metrics["mae"],4),
      "| RMSE:", round(metrics["rmse"],4), "| R2:", round(metrics["r2"],4),
      "| n_test:", metrics["n_test"])

imp = feature_importance_df(model)
print()
print(imp[["feature","gain_pct","split"]].to_string(index=False))
print()
print("YORUM:", interpret_importance(imp))

figs = get_paths(cfg)["figures_dir"]
plot_feature_importance(imp, figs)
plot_residuals(y_true, y_pred, figs)
print()
print("grafikler kaydedildi:", figs)

In [ ]:
from IPython.display import Image, display
import os
figs = str(get_paths(cfg)["figures_dir"])
display(Image(filename=os.path.join(figs, "feature_importance.png")))
display(Image(filename=os.path.join(figs, "residuals.png")))